# Partie 2 : Construction des pipelines de prétraitement

**Cours :** 030513 IFM, Intelligence artificielle appliquée au prétraitement des données (automne 2026)

**Évaluation sommative 2 :** Création des pipelines (Prof. Nadjate Saïdani)

**Équipe :**
- Akiella Wankoua, Blonde Béatrice
- Essomba, Martin Junior
- Gouater, Rene Eutyche
- Misse, Anthony

**But du notebook :** choisir un traitement pour chaque type de variable, construire les pipelines et les appliquer avec un ColumnTransformer.

**Ce que le notebook utilise :**
- `donnees/preparees/` (fichiers créés par le notebook 01)

**Ce que le notebook produit pour la suite :**
- `modeles/etape2_pretraitement.joblib` : les groupes de colonnes et les pipelines (lus par le notebook 03)

## Importation des librairies

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

DOSSIER_PREPAREES = Path("../donnees/preparees")
DOSSIER_MODELES = Path("../modeles")
DOSSIER_MODELES.mkdir(exist_ok=True)

## Chargement des données du notebook 01

In [2]:
# agent est un code : on le lit comme du texte
X_train = pd.read_csv(DOSSIER_PREPAREES / "X_train.csv", dtype={"agent": str})
X_test = pd.read_csv(DOSSIER_PREPAREES / "X_test.csv", dtype={"agent": str})
y_train = pd.read_csv(DOSSIER_PREPAREES / "y_train.csv")["is_canceled"]
y_test = pd.read_csv(DOSSIER_PREPAREES / "y_test.csv")["is_canceled"]

print("X_train :", X_train.shape, "| X_test :", X_test.shape)
print("y_train :", y_train.shape, "| y_test :", y_test.shape)

X_train : (69783, 27) | X_test : (17446, 27)
y_train : (69783,) | y_test : (17446,)


**Ce qu'on observe :**

- On retrouve les dimensions de la fin du notebook 01 : 69 783 lignes pour le train, 17 446 pour le test, 27 variables.

## 1. Définir les traitements selon le type de variable

In [3]:
# Nombre de modalités (valeurs différentes) de chaque variable catégorielle
colonnes_texte = X_train.select_dtypes(exclude="number").columns
X_train[colonnes_texte].nunique().sort_values()

hotel                     2
deposit_type              3
customer_type             4
distribution_channel      5
meal                      5
market_segment            8
reserved_room_type        9
arrival_date_month       12
country                 172
agent                   319
dtype: int64

**Ce qu'on observe :**

- 8 variables ont entre 2 et 12 modalités : un one-hot simple suffit.
- `country` (172 modalités) et `agent` (319 modalités) en ont beaucoup plus.
    - Un one-hot simple créerait 491 colonnes (172 + 319) pour ces deux variables.
    - Il faut regrouper les modalités rares.

Nous faisons 5 groupes de colonnes. Chaque groupe reçoit son propre traitement :

- **Numériques très asymétriques** : `lead_time`, `days_in_waiting_list`
    - Ce sont des durées en jours avec quelques valeurs très grandes.
    - Traitement : médiane pour les valeurs manquantes, puis log(1 + x), puis standardisation.
    - Le log rapproche les très grandes valeurs des autres valeurs.
- **Autres numériques** : comptages (adultes, nuits, demandes…), composantes de la date, prix `adr`
    - Traitement : médiane pour les valeurs manquantes, puis standardisation.
    - La standardisation met toutes les colonnes à la même échelle (moyenne 0, écart-type 1). C'est utile pour les modèles sensibles à l'échelle, comme la régression logistique.
    - On ne met pas `adr` au log : beaucoup de prix valent 0 et le log crée une asymétrie dans l'autre sens (voir les résultats plus bas).
- **Binaire** : `is_repeated_guest` est déjà codée 0 / 1, aucun traitement.
- **Catégorielles avec peu de modalités** (au plus 12)
    - Traitement : valeur la plus fréquente pour les valeurs manquantes, puis one-hot.
    - Le one-hot crée une colonne 0 / 1 par modalité, car les modèles ne lisent pas le texte.
- **Catégorielles avec beaucoup de modalités** : `country`, `agent`
    - Traitement : « Inconnu » pour les valeurs manquantes, puis one-hot.
    - Les modalités rares (moins de 0,5 % des lignes) sont regroupées dans une seule colonne « infrequent ». Sans ce regroupement, on aurait des centaines de colonnes presque vides.

Les imputeurs sont mis même dans les groupes sans valeur manquante : si une nouvelle réservation arrive avec une case vide, le pipeline saura quoi faire.

In [4]:
# a) Numériques très asymétriques -> médiane + log(1 + x) + standardisation
num_asymetriques = ["lead_time", "days_in_waiting_list"]

# b) Autres numériques -> médiane + standardisation
num_standard = [
    "adr",
    "arrival_date_year", "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies",
    "previous_cancellations", "previous_bookings_not_canceled",
    "booking_changes", "required_car_parking_spaces", "total_of_special_requests",
]

# c) Binaire déjà codée 0 / 1 -> aucun traitement
binaires = ["is_repeated_guest"]

# d) Catégorielles à peu de modalités -> valeur la plus fréquente + one-hot
cat_faible = [
    "hotel", "arrival_date_month", "meal", "market_segment",
    "distribution_channel", "reserved_room_type", "deposit_type", "customer_type",
]

# e) Catégorielles à beaucoup de modalités -> "Inconnu" + one-hot avec regroupement des modalités rares
cat_forte = ["country", "agent"]

# Vérification : chaque colonne est dans un seul groupe et aucune colonne n'est oubliée
toutes = num_asymetriques + num_standard + binaires + cat_faible + cat_forte
assert sorted(toutes) == sorted(X_train.columns), "Colonne oubliée ou en double"

groupes = {
    "num_asymetriques": num_asymetriques,
    "num_standard": num_standard,
    "binaires": binaires,
    "cat_faible": cat_faible,
    "cat_forte": cat_forte,
}
print("Nombre de colonnes par groupe :")
for nom, groupe in groupes.items():
    print(nom, ":", len(groupe))
print("Total :", len(toutes), "sur", X_train.shape[1], "colonnes")

Nombre de colonnes par groupe :
num_asymetriques : 2
num_standard : 14
binaires : 1
cat_faible : 8
cat_forte : 2
Total : 27 sur 27 colonnes


**Ce qu'on observe :**

- Les 27 colonnes sont réparties dans 5 groupes.
- La vérification `assert` passe : aucune colonne n'est oubliée et aucune n'est dans deux groupes.

## 2. Construire un pipeline pour chaque traitement

In [5]:
pipe_num_asym = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler()),
])

pipe_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

pipe_cat_faible = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# min_frequency=0.005 : une modalité présente dans moins de 0,5 % des lignes va dans "infrequent"
# handle_unknown="infrequent_if_exist" : une modalité jamais vue dans le train va aussi dans "infrequent"
pipe_cat_forte = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Inconnu")),
    ("onehot", OneHotEncoder(
        min_frequency=0.005,
        handle_unknown="infrequent_if_exist",
        sparse_output=False,
    )),
])

## 3. Appliquer les pipelines avec ColumnTransformer

In [6]:
preprocesseur = ColumnTransformer(
    transformers=[
        ("num_asym", pipe_num_asym, num_asymetriques),
        ("num", pipe_num, num_standard),
        ("bin", "passthrough", binaires),
        ("cat_faible", pipe_cat_faible, cat_faible),
        ("cat_forte", pipe_cat_forte, cat_forte),
    ],
    remainder="drop",
)

# fit sur le TRAIN seulement (médianes, moyennes, écarts-types, modalités),
# puis transform sur le test : le test ne sert pas à apprendre, donc pas de fuite d'information
X_train_prep = preprocesseur.fit_transform(X_train)
X_test_prep = preprocesseur.transform(X_test)

noms = preprocesseur.get_feature_names_out()
X_train_prep = pd.DataFrame(X_train_prep, columns=noms)
X_test_prep = pd.DataFrame(X_test_prep, columns=noms)

## 4. Résultats après prétraitement

In [7]:
print("Dimensions :")
print("X_train :", X_train.shape, "->", X_train_prep.shape)
print("X_test :", X_test.shape, "->", X_test_prep.shape)

print()
print("Nombre de colonnes produites par groupe :")
print(pd.Series([n.split("__")[0] for n in noms]).value_counts().to_string())

Dimensions :
X_train : (69783, 27) -> (69783, 105)
X_test : (17446, 27) -> (17446, 105)

Nombre de colonnes produites par groupe :
cat_faible    48
cat_forte     40
num           14
num_asym       2
bin            1


**Ce qu'on observe :**

- Après le prétraitement, on passe de 27 à 105 colonnes.
- La hausse vient du one-hot :
    - `cat_faible` produit 48 colonnes ;
    - `cat_forte` produit 40 colonnes.
- Les colonnes numériques gardent leur nombre (2 + 14) et la colonne binaire reste seule.
- Le test a le même nombre de colonnes que le train (105). C'est obligatoire pour que le modèle puisse prédire.

In [8]:
print("Valeurs manquantes AVANT le pipeline (train) :", int(X_train.isna().sum().sum()))
print(X_train.isna().sum()[X_train.isna().sum() > 0].to_string())
print()
print("Valeurs manquantes APRÈS le pipeline (train) :", int(X_train_prep.isna().sum().sum()))
print("Valeurs manquantes APRÈS le pipeline (test) :", int(X_test_prep.isna().sum().sum()))

Valeurs manquantes AVANT le pipeline (train) : 365


country    365

Valeurs manquantes APRÈS le pipeline (train) : 0
Valeurs manquantes APRÈS le pipeline (test) : 0


**Ce qu'on observe :**

- Avant le pipeline, il y a 365 valeurs manquantes dans le train, toutes dans `country`.
- Après le pipeline, il n'en reste aucune, ni dans le train, ni dans le test.

In [9]:
print("Asymétrie (skewness) sur le train, avant et après le pipeline :")
for colonne in num_asymetriques:
    avant = X_train[colonne].skew()
    apres = X_train_prep["num_asym__" + colonne].skew()
    print(f"{colonne} : avant {avant:.2f}, après {apres:.2f}")

# Pourquoi adr n'est pas dans le groupe "log"
adr_avant = X_train["adr"].skew()
adr_si_log = np.log1p(X_train["adr"]).skew()
print(f"adr (log non retenu) : avant {adr_avant:.2f}, si log {adr_si_log:.2f}")

Asymétrie (skewness) sur le train, avant et après le pipeline :
lead_time : avant 1.44, après -0.72
days_in_waiting_list : avant 19.75, après 10.81
adr (log non retenu) : avant 13.09, si log -3.58


**Ce qu'on observe :**

- Une asymétrie proche de 0 veut dire une distribution plus équilibrée.
- `lead_time` passe de 1,44 à -0,72 : le log a bien réduit la longue queue à droite.
- `days_in_waiting_list` passe de 19,75 à 10,81.
    - L'asymétrie baisse, mais reste forte.
    - La raison : la valeur est 0 pour la plupart des réservations (voir `describe` au notebook 01). Le log ne peut pas changer ça.
- Pour `adr`, le log ferait passer l'asymétrie de 13,09 à -3,58 : elle changerait de côté sans devenir petite. C'est pour ça qu'on ne met pas `adr` au log. Ses valeurs extrêmes seront traitées au notebook 03.

In [10]:
# Après standardisation : moyenne proche de 0 et écart-type proche de 1 (train)
colonnes_num = [n for n in noms if n.startswith(("num__", "num_asym__"))]
X_train_prep[colonnes_num].agg(["mean", "std"]).T.round(3)

,mean,std
num_asym__lead_time,0.0,1.0
num_asym__days_in_waiting_list,0.0,1.0
num__adr,-0.0,1.0
num__arrival_date_year,0.0,1.0
num__arrival_date_week_number,0.0,1.0
num__arrival_date_day_of_month,0.0,1.0
num__stays_in_weekend_nights,0.0,1.0
num__stays_in_week_nights,0.0,1.0
num__adults,-0.0,1.0
num__children,0.0,1.0


**Ce qu'on observe :**

- Les 16 colonnes numériques ont maintenant une moyenne de 0 et un écart-type de 1 sur le train.
- Elles sont toutes à la même échelle. Par exemple, l'année (2015 à 2017) et le nombre de bébés (0 à 10, voir le notebook 01) partent maintenant avec le même poids.

In [11]:
ohe_forte = preprocesseur.named_transformers_["cat_forte"].named_steps["onehot"]

for variable, modalites, rares in zip(cat_forte, ohe_forte.categories_, ohe_forte.infrequent_categories_):
    nb_rares = 0 if rares is None else len(rares)
    nb_gardees = len(modalites) - nb_rares
    texte = f"{variable} : {len(modalites)} modalités -> {nb_gardees} gardées"
    print(texte, f"+ 1 colonne « infrequent » ({nb_rares} modalités regroupées)")

country : 173 modalités -> 21 gardées + 1 colonne « infrequent » (152 modalités regroupées)
agent : 319 modalités -> 17 gardées + 1 colonne « infrequent » (302 modalités regroupées)


**Ce qu'on observe :**

- `country` : on garde 21 modalités et les 152 autres sont regroupées dans la colonne « infrequent ».
- `agent` : on garde 17 modalités et les 302 autres sont regroupées.
- Les deux variables produisent donc 40 colonnes au lieu de 492 (173 + 319).
- Un pays ou un agent jamais vu dans le train ira aussi dans « infrequent ». Le pipeline ne plantera pas avec de nouvelles réservations.

In [12]:
# Aperçu des 5 premières lignes et des 8 premières colonnes après prétraitement
X_train_prep.iloc[:5, :8].round(2)

,num_asym__lead_time,num_asym__days_in_waiting_list,num__adr,num__arrival_date_year,num__arrival_date_week_number,num__arrival_date_day_of_month,num__stays_in_weekend_nights,num__stays_in_week_nights
0,0.75,-0.1,1.89,1.15,0.08,-0.55,0.97,1.17
1,0.08,-0.1,0.82,-0.31,-0.43,-0.09,0.97,1.17
2,0.45,-0.1,0.04,1.15,-1.23,-0.77,-0.00,0.19
3,0.72,-0.1,-0.78,1.15,-1.23,-1.00,-0.98,-0.31
4,-0.75,-0.1,-0.46,-0.31,-1.45,-0.43,-0.98,-0.31


**Ce qu'on observe :**

- Les valeurs sont maintenant centrées autour de 0, positives ou négatives.
- Exemple : la ligne 0 a `num__adr` = 1,89. Son prix est donc plus élevé que la moyenne du train, de 1,89 écart-type.
- Le nom de chaque colonne commence par le nom de son groupe (`num_asym__`, `num__`…). On sait ainsi quel traitement elle a reçu.

## Sauvegarde pour le notebook 03

In [13]:
# On sauvegarde les groupes de colonnes et les pipelines (non entraînés).
# Le notebook 03 va les réutiliser et ajouter le transformateur personnalisé.
etape2 = {
    "groupes": {
        "num_asym": num_asymetriques,
        "num": num_standard,
        "bin": binaires,
        "cat_faible": cat_faible,
        "cat_forte": cat_forte,
    },
    "pipelines": {
        "num_asym": pipe_num_asym,
        "num": pipe_num,
        "cat_faible": pipe_cat_faible,
        "cat_forte": pipe_cat_forte,
    },
}

joblib.dump(etape2, DOSSIER_MODELES / "etape2_pretraitement.joblib")
print("Fichier créé : modeles/etape2_pretraitement.joblib")

Fichier créé : modeles/etape2_pretraitement.joblib


**Ce qu'on observe :**

- Le fichier contient les 5 groupes de colonnes et les 4 pipelines.
- Le notebook 03 va les réutiliser sans les réécrire.